# PerturbGen end-to-end (Python API): perturb at 6 h, predict 10 h

> **Note:** this tutorial is the Python-API version of the PerturbGen workflow; results may differ slightly from those reported in the paper.

This example trains PerturbGen on the in vivo LPS challenge with the **6 h** state as the source and **10 h** as the target, then knocks out **IL1B** at 6 h and predicts how the 10 h state changes.

API — **tokenized data → train (masking, then count) → embeddings → perturb**, with `save`/`load`. Everything runs in-process (no shell/`subprocess`).

```python
import perturbgen as pg

# 1) TOKENIZE your preprocessed AnnData (CPU; no encoder needed):
#    tokenized_dir = pg.tokenize(adata_path, dataset='lps_6h_perturb',
#                                reference_time='6h_LPS', time_point_order=['6h_LPS', '10h_LPS'], ...)

# 2) BUILD THE MODEL from the tokenized data.
#    The pretrained scmaskgit encoder shipped with the repository is used by default:
model = pg.PerturbGen(tokenized_dir, source='6h_LPS', pred_tps=[1])

# 3) TRAIN in two steps so you pick which masking checkpoint feeds the decoder:
masking_ckpt = model.train_masking(output_dir='model')
count_ckpt   = model.train_count(output_dir='model', masking_ckpt=masking_ckpt)

# 4) SAVE / EMBED / PERTURB (checkpoints are reused; load() restores them):
model.save('model')                                    # pg.PerturbGen.load('model')
emb  = model.get_embeddings(masking_ckpt=masking_ckpt)
pred = model.perturb(genes=['ENSG00000125538'], count_ckpt=count_ckpt, pred_tps=[1])
```

## Prerequisites

- A **preprocessed AnnData** (QC'd, gene IDs harmonised — see the preprocessing notebook).
- The **tokenizer reference files** shipped with PerturbGen (`gene_median`, `token_dict`, `gene_mapping`).
- The **pretrained `scmaskgit` encoder** checkpoint (the model's frozen foundation backbone — a *model* input, **not** used by tokenization). It is used by default from `pretraining_cohort/` in the repository; if `git lfs pull` fails, download it from Hugging Face and save it there (see README).
- A **CUDA GPU** for `train`/`get_embeddings`/`perturb`. `batch_size=64` fits 24 GB GPUs (measured peak ~16 GB); lower it if you hit an out-of-memory error.

In [ ]:
import glob
import os

import perturbgen as pg
from perturbgen.configs import GENE_MAPPING_PATH, GENE_MEDIAN_PATH, TOKEN_DICT_PATH

# Path to the preprocessed LPS AnnData (raw counts in .X, Ensembl IDs as the var index).
# Download it with:
#   aws --endpoint-url https://cog.sanger.ac.uk --no-sign-request s3 cp s3://perturbgen/Manuscript/lps_otar.h5ad .
ADATA_PATH = 'path/to/lps_otar.h5ad'

# Tokenizer reference dictionaries shipped with PerturbGen (perturbgen/pp/).
GENE_MEDIAN = str(GENE_MEDIAN_PATH)
TOKEN_DICT = str(TOKEN_DICT_PATH)
GENE_MAPPING = str(GENE_MAPPING_PATH)

# The pretrained scmaskgit encoder is used by default from pretraining_cohort/ in the repository
# (the same file as on Hugging Face); pass encoder_path= to pg.PerturbGen to use another one.

# Tokenized-data location, defined here so TOKENIZED_DIR / SOURCE survive a kernel restart
# without re-running the tokenize cell (the tokenize cell below builds the data at this path).
DATASET_NAME = 'lps_6h_perturb'
from perturbgen.configs.paths import TOKENIZED_DIR as _TOK_ROOT
TOKENIZED_DIR = os.path.join(str(_TOK_ROOT), DATASET_NAME)
SOURCE = '6h_LPS'

## 1. Tokenize your data

Tokenize + pair a **preprocessed AnnData** (raw counts in `.X`, Ensembl IDs as the `var` index) with `pg.tokenize(...)` — a CPU step that writes the `.dataset` layout the model reads. `reference_time` is the **source** state (pairs run *source → later `time_point_order`*); here the source is `6h_LPS` and the only target is `10h_LPS`.

Gene selection defaults to the **released recipe**: `hvg_flavor='seurat_v3'` (raw-count variance ranking) over the Geneformer-tokenizable genes, batched by `time_obs`.

In [3]:
# Tokenize a preprocessed AnnData -> writes the .dataset layout the model reads.
# Defaults reproduce the released tokenisation (seurat_v3 HVG over the GF-tokenizable
# pool; cell/mito filters off). Point ADATA_PATH (cell above) at your data.
TOKENIZED_DIR = pg.tokenize(
    ADATA_PATH,
    dataset=DATASET_NAME,
    reference_time='6h_LPS',                           # source state
    time_obs='time_after_LPS',                         # also the HVG batch key
    time_point_order=['6h_LPS', '10h_LPS'],            # source -> target
    var_list=['cell_pairing_index', 'time_after_LPS', 'cell_type_harmonized'],
    main_pairing_obs='cell_type_harmonized',
    gene_median_path=GENE_MEDIAN,
    token_dict_path=TOKEN_DICT,
    gene_mapping_path=GENE_MAPPING,
    n_hvg=2000,
    hvg_mode='after_tokenisation',
    hvg_flavor='seurat_v3',        # raw-count variance ranking (release recipe)
    exclude_non_gf_genes=True,     # HVG over the Geneformer-tokenizable genes
    cell_gene_filter=False,
    remove_mito_ribo_genes=False,
    nproc=4,
)

[PerturbGen] tokenizing /nfs/team361/am74/Cytomeister/Evaluation_datasets/LPS/full_lps_new.h5ad -> dataset 'lps_6h_perturb' ...
Current working directory: /lustre/scratch126/cellgen/lotfollahi/dv8
Start preprocessing adata...
Number of genes dropped: 0
Finished preprocessing adata.
Start tokenisation of adata...
Tokenizing /lustre/scratch126/cellgen/lotfollahi/dv8/T_perturb/tokenized_data/lps_6h_perturb/h5ad_pairing_2000_hvg/lps_6h_perturb.h5ad


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/perturbgen/pp/tokenizer.py:495: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/perturbgen/pp/tokenizer.py:498: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/lustre/scratch126/cellgen/lotfollahi/dv8/T_perturb/tokenized_data/lps_6h_perturb/h5ad_pairing_2000_hvg/lps_6h_perturb.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


Map (num_proc=4):   0%|          | 0/224150 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/224150 [00:00<?, ? examples/s]

...

  0%|                                                                                                                                                                                                                                                              | 0/2 [00:00<?, ?it/s]/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Saving the dataset (0/1 shards):   0%|          | 0/148107 [00:00<?, ? examples/s]

 50%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                                                                                                                           | 1/2 [00:05<00:05,  5.97s/it]/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Saving the dataset (0/1 shards):   0%|          | 0/148107 [00:00<?, ? examples/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:11<00:00,  5.71s/it]


## 2. Build the model

`PerturbGen(...)` reads the already-tokenized data (infers the tokenizer's file layout from the directory, `source`, `n_hvg`). The pretrained encoder defaults to the one shipped with the repository (pass `encoder_path=` to use another); architecture and retained columns (`var_list`) are set here.

In [ ]:
model = pg.PerturbGen(
    TOKENIZED_DIR,
    source=SOURCE,
    n_hvg=2000,
    n_layers=6, d_ff=32, d_model=768,          # LPS: d_ff=32
    pred_tps=[1],                              # 6h source -> target 10h
    var_list=['cell_type_harmonized', 'cell_pairing_index', 'time_after_LPS'],
    cond_list=['time_after_LPS'],              # time conditioning
    mask_scheduler='cosine',                   # API default is 'pow'
    context_mode=True,                         # ON during training (you can adjust)
    seed=42,
)

## 3a. Train the masking model (GPU required)

A checkpoint is saved **for every epoch** under `lps_model/masking/checkpoints`. Inspect the training curves (e.g. in Weights & Biases) to decide which epoch to feed the count decoder. `train_masking` returns the **metric-best** checkpoint (the default used by `train_count`).

In [4]:
masking_ckpt = model.train_masking(
    output_dir='lps_model',
    max_epochs=10,
    batch_size=64,   # fits 24 GB GPUs (measured peak ~16 GB); lower if out of memory
    cellgen_lr=1e-4, cellgen_wd=1e-4,
    seed=0,
)
print('best masking checkpoint:', masking_ckpt)

[PerturbGen] training masking model ...


Seed set to 42


Current working directory: /lustre/scratch126/cellgen/lotfollahi/dv8
Loading and preprocessing data...
Loading 1_10h_LPS.dataset...
Loading 1_10h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


---PerturbGen training --- 
Target vocab size: 2004, max sequence length: 751
Using NVIDIA H100 80GB HBM3 for training
Set float32_matmul_precision to medium
-- Initializing scmaskgit model


...

Sanity Checking: |                                                                                            …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:105: Total length of `list` across ranks is zero. Please make sure this was your intention.


Training: |                                                                                                   …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('lr', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/perplexity', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/masking_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('lr', ..., logger=True)` but have no 

[PerturbGen] 20 masking checkpoint(s) under /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints
[PerturbGen] metric-best (default for train_count): /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260907_1911_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1_s_0-epoch=09.ckpt
best masking checkpoint: /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260907_1911_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1_s_0-epoch=09.ckpt


## 3b. Train the count decoder (GPU required)

By default `train_count` uses the metric-best masking checkpoint from the previous step. To use a **different epoch** (based on the masking curves), pass it explicitly, as done here (masking epoch 8, then count epoch 2 for the perturbation).

In [5]:
# This example initialises the count decoder from masking epoch 8 and perturbs with count epoch 2.
count_ckpt = model.train_count(
    output_dir='lps_model',
    masking_ckpt=sorted(glob.glob('lps_model/masking/checkpoints/*epoch=08.ckpt'))[-1],
    max_epochs=4,
    count_lr=1e-3, count_wd=1e-3, count_dropout=0.1,
    seed=42,
)
count_ckpt = sorted(glob.glob('lps_model/count/checkpoints/*epoch=02.ckpt'))[-1]
model.count_ckpt = count_ckpt   # keep this epoch for save()/load() and perturb()
print('count decoder checkpoint:', count_ckpt)

Seed set to 42


[PerturbGen] training count decoder from /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260907_1911_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1_s_0-epoch=08.ckpt ...
Loading and preprocessing data...
Loading 1_10h_LPS.dataset...
Loading 1_10h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


---PerturbGen training --- 
Target vocab size: 2004, max sequence length: 751
Using NVIDIA H100 80GB HBM3 for training
Set float32_matmul_precision to medium
-- Initializing scmaskgit model


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/perturbgen/Model/trainer.py:674: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_masking_path

...

Sanity Checking: |                                                                                            …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:105: Total length of `list` across ranks is zero. Please make sure this was your intention.


Training: |                                                                                                   …

/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/core/module.py:516: You called `self.log('train/mse', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 64. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from

count decoder checkpoint: /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/count/checkpoints/20260907_2047_cellgen_train_count_lr_0.001_wd_0.001_batch_64_drop_0.1_zinb_tp_1_s_42_pos_time_pos_sin_m_cosine-epoch=03.ckpt


## 4. Save / load

Saves a manifest (tokenized dir, source, hyper-parameters, **and the masking + count checkpoint paths**). After `load()` the checkpoints are restored, so you can extract embeddings or perturb **without retraining**.

In [ ]:
model.save('lps_model')
# later:  model = pg.PerturbGen.load('lps_model')

## 5. Gene / cell embeddings

Embeddings come from the **masking model**, so this uses a **masking checkpoint** (defaults to the trained one, or `model.masking_ckpt` after `load()`; pass `masking_ckpt=` to choose a specific epoch). Output feeds gene-program discovery (`gene_program_discovery` notebook); for the paper's gene-program analysis use the normal-source embeddings from the `cell_gene_embeddings` notebook.

In [8]:
emb_dir = model.get_embeddings(
    output_dir='lps_embeddings',
    masking_ckpt=masking_ckpt,   # masking checkpoint (embeddings come from the masking model)
    return_gene_embs=True,
    return_cell_embs=True,
    gene_embs_condition='time_after_LPS',   # gene embs are extracted per unique value of this column
)
print('embeddings at:', emb_dir)

Seed set to 42


[PerturbGen] extracting embeddings -> /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_embeddings ...
Current working directory: /lustre/scratch126/cellgen/lotfollahi/dv8
positional encoding: time_pos_sin
Loading and preprocessing data...
Loading 2_10h_LPS.dataset...
Loading 1_6h_LPS.dataset...
Loading 1_6h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_10h_LPS.h5ad...


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


...

Restoring states from the checkpoint path at /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=09.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6]
Loaded model weights from the checkpoint at /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_model/masking/checkpoints/20260901_1827_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_cosine_tp_1-2_s_0-epoch=09.ckpt


Testing: |                                                                                                    …

---Start saving embeddings


/lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


End saving embeddings---
embeddings at: /lustre/scratch126/cellgen/lotfollahi/dv8/Perturbgen/lps_embeddings


## 6. In-silico perturbation

Perturbation prediction uses the trained **count-decoder checkpoint** (defaults to the trained one, or `model.count_ckpt` after `load()`; pass `count_ckpt=` to use a specific one).

- `mode`: `'knockout'` (mask) | `'overexpress'` | `'delete'` | `'pad'`.
- `where`: `'source'` (edit the source state, propagate to `pred_tps`) | `'target'`.
- Pass a **list** of genes to knock several out **together**.

> Use a fresh `output_dir` so previously-computed genes aren't skipped.

In [ ]:
pred_dir = model.perturb(
    genes=['ENSG00000125538'],   # IL1B, knocked out in the 6h source cells
    count_ckpt=count_ckpt,       # trained count-decoder checkpoint
    mode='knockout', where='source',
    pred_tps=[1], n_samples=3,   # predict the 10h state
    temperature=1.5, iterations=19,   # LPS config (MaskGIT sampling; code defaults 2.0/18)
    output_dir='lps_6h_perturbation',  # separate from the perturbation notebook (released IL1B model)
)
print('predicted post-perturbation h5ads at:', pred_dir)

## Next steps

- Predicted post-perturbation AnnDatas → **post-perturbation analysis** (`post_perturbation_analyses` notebook: DEGs, program shifts, trajectory effects).
- Embedding outputs → **gene-program discovery** (`gene_program_discovery` notebook).

> Note: the paper's *cell/gene-embedding* analyses use the `normal`-source model; the `cell_gene_embeddings` notebook extracts them from the released checkpoint.